# TCN Hyperparameter Optimization — MIT-BIH ECG Forecasting

**Author:** Ibrahim Hanafy  
**Date:** August 2026  
**Objective:** Systematic hyperparameter optimization using Optuna (TPE + pruning) to find the best TCN configuration for multi-step ECG forecasting.

**Baseline (Our Config):** 3 blocks, 64 filters, kernel=3, dropout=0.1, lr=1e-3, batch=256  
**Search goal:** Beat baseline RMSE across all horizons (H=1, 10, 15, 20)

---

## 1 — Imports & Setup

In [ ]:
import os, time, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [ ]:
# ── TensorFlow Setup ─────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, Add, Activation, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Optuna Setup ──────────────────────────────────────────────────────────────
!pip install -q optuna
import optuna
from optuna.exceptions import TrialPruned

optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f'Optuna {optuna.__version__}')

## 2 — Constants & Dataset

In [ ]:
# ── Dataset path ──────────────────────────────────────────────────────────────
DATA_DIR = r"/kaggle/input/datasets/rracer17/mit-bih-mitdb/mit-bih-arrhythmia-database-1.0.0"

# ── Paper constants (Section 3) ──────────────────────────────────────────────
FS           = 360
TOTAL_STEPS  = 100_000
TRAIN_STEPS  = 40_000
VAL_STEPS    = 10_000
TEST_STEPS   = 50_000
LOOKBACK     = 10

# ── All 21 patients (excluding 111 & 118) ───────────────────────────────────
PATIENTS = [
    '100', '101', '102', '103', '104', '105',
    '106', '107', '108', '109', '112', '113',
    '114', '115', '116', '117', '119',
    '121', '122', '123', '124'
]
assert len(PATIENTS) == 21

HORIZONS = [1, 10, 15, 20]

# ── Pilot patients for HPO (diverse morphologies + difficulty levels) ────────
# Picked from our baseline results: mix of easy (100,124), medium (103,119),
# and hard (108) patients to ensure the search generalises.
PILOT_PATIENTS = ['100', '103', '108', '119', '124']

# ── Pilot horizon (mid-range proxy — balances short & long horizon perf) ────
PILOT_HORIZON = 10

print(f'Full patients : {len(PATIENTS)}')
print(f'Pilot patients: {len(PILOT_PATIENTS)} → {PILOT_PATIENTS}')
print(f'Pilot horizon : H={PILOT_HORIZON}')

## 3 — Data Pipeline

Identical to baseline notebooks — load, split, normalise, create sequences.

In [ ]:
def load_ecg_signal(record_id, data_dir, n_steps=100_000):
    """Load MLII lead from MIT-BIH, truncate/pad to n_steps."""
    path = os.path.join(data_dir, record_id)
    rec  = wfdb.rdrecord(path)
    sig_names_upper = [s.upper() for s in rec.sig_name]
    ch = sig_names_upper.index('MLII') if 'MLII' in sig_names_upper else 0
    signal = rec.p_signal[:, ch].astype(np.float32)
    if len(signal) < n_steps:
        pad = np.full(n_steps - len(signal), signal[-1], dtype=np.float32)
        signal = np.concatenate([signal, pad])
    return signal[:n_steps]


def preprocess_patient(signal, train_steps=40_000, val_steps=10_000):
    """Split → train/val/test, MinMax-normalise (fit on train only)."""
    train_end = train_steps
    val_end   = train_steps + val_steps
    train_raw = signal[:train_end]
    val_raw   = signal[train_end:val_end]
    test_raw  = signal[val_end:]
    scaler     = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train_raw.reshape(-1, 1)).flatten()
    val_norm   = scaler.transform(val_raw.reshape(-1, 1)).flatten()
    test_norm  = scaler.transform(test_raw.reshape(-1, 1)).flatten()
    return train_norm, val_norm, test_norm, scaler


def make_multistep_sequences(signal, lookback, horizon):
    """X = lookback window, y = next H steps (Multi-Output)."""
    X, y = [], []
    for i in range(len(signal) - lookback - horizon + 1):
        X.append(signal[i : i + lookback])
        y.append(signal[i + lookback : i + lookback + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def compute_metrics(y_true, y_pred):
    yt, yp = y_true.flatten(), y_pred.flatten()
    return {
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'MAE':  float(mean_absolute_error(yt, yp)),
        'R2':   float(r2_score(yt, yp)),
    }


def compute_per_step_metrics(y_true, y_pred):
    rows = []
    for h in range(y_true.shape[1]):
        m = compute_metrics(y_true[:, h], y_pred[:, h])
        m['Step'] = h + 1
        rows.append(m)
    return pd.DataFrame(rows)


print('Data pipeline functions defined.')

In [ ]:
# ── Load ALL patients (needed for both HPO and final evaluation) ──────────────
patient_signals = {}

print(f'Loading {len(PATIENTS)} patients...\n')
for rid in tqdm(PATIENTS, desc='Patients'):
    signal = load_ecg_signal(rid, DATA_DIR, n_steps=TOTAL_STEPS)
    tr, vl, te, sc = preprocess_patient(signal, TRAIN_STEPS, VAL_STEPS)
    patient_signals[rid] = {'train': tr, 'val': vl, 'test': te, 'scaler': sc}
    tqdm.write(f'  Patient {rid:>3s} | train={len(tr):,}  val={len(vl):,}  test={len(te):,}')

print(f'\nAll {len(patient_signals)} patients loaded.')

## 4 — Model Architecture

Same residual block as baseline. `build_tcn` now accepts all hyperparameters as arguments instead of globals.

In [ ]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(out)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1)(x)
    return Add()([x, out])


def build_tcn(lookback, output_size, n_blocks, n_filters, kernel_size, dropout_rate, learning_rate):
    """Build TCN with explicit hyperparameters (no globals)."""
    inp = Input(shape=(lookback, 1))
    x = inp
    for i in range(n_blocks):
        x = residual_block(x, n_filters, kernel_size, 2 ** i, dropout_rate)
    x = x[:, -1, :]  # last time-step
    x = Dense(n_filters, activation='relu')(x)
    out = Dense(output_size)(x)
    model = Model(inp, out, name=f'TCN_out{output_size}')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss='mse')
    return model


print('Model architecture defined.')

## 5 — Optuna Objective Function

**Strategy:**
- Train on 5 pilot patients at H=10 (mid-range proxy)
- 100 epochs max with patience=20 (faster than full 200/50)
- Pruning via val_loss monitoring — kills bad trials early
- Objective = mean RMSE across pilot patients

In [ ]:
def objective(trial):
    """Optuna objective: train TCN on pilot patients, return mean RMSE."""

    # ── Suggest hyperparameters ───────────────────────────────────────────────
    n_blocks    = trial.suggest_int('n_blocks', 2, 6)
    n_filters   = trial.suggest_categorical('n_filters', [32, 64, 128, 256])
    kernel_size = trial.suggest_categorical('kernel_size', [2, 3, 5, 7])
    dropout     = trial.suggest_float('dropout', 0.05, 0.4, step=0.05)
    lr          = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size  = trial.suggest_categorical('batch_size', [64, 128, 256, 512])

    rmse_scores = []

    for pid in PILOT_PATIENTS:
        tf.keras.backend.clear_session()
        gc.collect()

        tr = patient_signals[pid]['train']
        vl = patient_signals[pid]['val']
        te = patient_signals[pid]['test']

        X_tr, y_tr = make_multistep_sequences(tr, LOOKBACK, PILOT_HORIZON)
        X_vl, y_vl = make_multistep_sequences(vl, LOOKBACK, PILOT_HORIZON)
        X_te, y_te = make_multistep_sequences(te, LOOKBACK, PILOT_HORIZON)

        # Reshape for Conv1D
        X_tr_r = X_tr.reshape(-1, X_tr.shape[1], 1)
        X_vl_r = X_vl.reshape(-1, X_vl.shape[1], 1)
        X_te_r = X_te.reshape(-1, X_te.shape[1], 1)

        # Build model
        model = build_tcn(LOOKBACK, PILOT_HORIZON,
                          n_blocks, n_filters, kernel_size, dropout, lr)

        # Train with early stopping
        history = model.fit(
            X_tr_r, y_tr,
            validation_data=(X_vl_r, y_vl),
            epochs=100,
            batch_size=batch_size,
            callbacks=[
                EarlyStopping(monitor='val_loss', patience=20,
                              restore_best_weights=True),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                  patience=10, min_lr=1e-6),
            ],
            verbose=0
        )

        # Evaluate
        preds = model.predict(X_te_r, batch_size=2048, verbose=0)
        rmse = float(np.sqrt(mean_squared_error(y_te.flatten(), preds.flatten())))
        rmse_scores.append(rmse)

        del model
        gc.collect()

    mean_rmse = np.mean(rmse_scores)

    # Report intermediate value for pruning
    trial.report(mean_rmse, step=0)
    if trial.should_prune():
        raise TrialPruned()

    return mean_rmse


print('Objective function defined.')
print(f'Each trial trains on {len(PILOT_PATIENTS)} patients × H={PILOT_HORIZON}')

## 6 — Run Hyperparameter Search

**Budget:** 50 trials (adjustable). TPE sampler learns from past trials.  
**Storage:** SQLite DB — study survives kernel restarts.  
**Time estimate:** ~2–4 hours depending on GPU.

In [ ]:
# ┌──────────────────────────────────────────────────────────────────────────┐
# │  CONFIGURATION — adjust these before running                           │
# └──────────────────────────────────────────────────────────────────────────┘
N_TRIALS     = 50       # total trials (increase for better coverage)
TIMEOUT      = 4 * 3600 # max seconds (4 hours safety cap)
STUDY_NAME   = 'tcn_ecg_hpo_v1'
DB_PATH      = 'sqlite:///tcn_hpo.db'

In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='minimize',
    storage=DB_PATH,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,   # random trials before pruning kicks in
        n_warmup_steps=0,
    )
)

print(f'Study "{STUDY_NAME}" created/loaded.')
print(f'Existing trials: {len(study.trials)}')
print(f'Running {N_TRIALS} trials with {TIMEOUT}s timeout...\n')

study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT, show_progress_bar=True)

print(f'\n{"=" * 60}')
print(f'Search complete!')
print(f'Total trials : {len(study.trials)}')
print(f'Best RMSE    : {study.best_value:.6f}')
print(f'Best params  :')
for k, v in study.best_params.items():
    print(f'  {k:>12s} = {v}')
print(f'{"=" * 60}')

## 7 — Analyse Search Results

In [ ]:
# ── Trial history DataFrame ──────────────────────────────────────────────────
df_trials = study.trials_dataframe()
df_trials = df_trials.sort_values('value')
print(f'Top 10 trials by RMSE:\n')
display(df_trials[['number', 'value', 'params_n_blocks', 'params_n_filters',
                    'params_kernel_size', 'params_dropout', 'params_lr',
                    'params_batch_size', 'duration']].head(10))

In [ ]:
# ── Parameter Importance ─────────────────────────────────────────────────────
fig = optuna.visualization.plot_param_importances(study)
fig.update_layout(title='Hyperparameter Importance (fANOVA)', height=400)
fig.show()

In [ ]:
# ── Optimization History ─────────────────────────────────────────────────────
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(title='Optimization History', height=400)
fig.show()

In [ ]:
# ── Parallel Coordinate Plot ─────────────────────────────────────────────────
fig = optuna.visualization.plot_parallel_coordinate(
    study, params=['n_blocks', 'n_filters', 'kernel_size', 'dropout', 'lr']
)
fig.update_layout(title='Parallel Coordinate (hyperparams → RMSE)', height=500)
fig.show()

In [ ]:
# ── Contour Plots for top parameter pairs ────────────────────────────────────
for p1, p2 in [('n_blocks', 'n_filters'), ('lr', 'dropout'), ('kernel_size', 'n_blocks')]:
    fig = optuna.visualization.plot_contour(study, params=[p1, p2])
    fig.update_layout(title=f'Contour: {p1} vs {p2}', height=450)
    fig.show()

## 8 — Full Evaluation with Best Config

Take the best hyperparameters from the search and run the **complete experiment:**
- All 21 patients
- All 4 horizons (H=1, 10, 15, 20)
- 200 epochs, patience 50 (full training budget)

In [ ]:
# ┌──────────────────────────────────────────────────────────────────────────┐
# │  BEST CONFIG FROM SEARCH                                               │
# └──────────────────────────────────────────────────────────────────────────┘
best = study.best_params

CONFIG_NAME   = 'Optimized'
EPOCHS        = 200
PATIENCE      = 50
BATCH_SIZE    = best['batch_size']
NUM_FILTERS   = best['n_filters']
KERNEL_SIZE   = best['kernel_size']
NUM_BLOCKS    = best['n_blocks']
DROPOUT_RATE  = best['dropout']
LEARNING_RATE = best['lr']

CSV_NAME = 'tcn_mo_optimized_results.csv'
PLOT_DIR = 'plots_optimized'
os.makedirs(PLOT_DIR, exist_ok=True)

print(f'Config    : {CONFIG_NAME}')
print(f'Patients  : {len(PATIENTS)}')
print(f'Horizons  : {HORIZONS}')
print(f'TCN       : blocks={NUM_BLOCKS}, filters={NUM_FILTERS}, kernel={KERNEL_SIZE}')
print(f'Dropout   : {DROPOUT_RATE}')
print(f'Training  : epochs={EPOCHS}, patience={PATIENCE}, batch={BATCH_SIZE}, lr={LEARNING_RATE:.6f}')

In [ ]:
# ── Full Experiment Loop ─────────────────────────────────────────────────────
results          = []
per_step_results = {}
saved_preds      = {}
training_histories = {}

total_runs = len(PATIENTS) * len(HORIZONS)
print(f'Running: {len(PATIENTS)} patients × {len(HORIZONS)} horizons = {total_runs} models')
print('=' * 80)

for rid in tqdm(PATIENTS, desc='Patients'):
    tr = patient_signals[rid]['train']
    vl = patient_signals[rid]['val']
    te = patient_signals[rid]['test']

    for horizon in HORIZONS:
        X_tr, y_tr = make_multistep_sequences(tr, LOOKBACK, horizon)
        X_vl, y_vl = make_multistep_sequences(vl, LOOKBACK, horizon)
        X_te, y_te = make_multistep_sequences(te, LOOKBACK, horizon)

        t0 = time.time()
        model = build_tcn(LOOKBACK, horizon,
                          NUM_BLOCKS, NUM_FILTERS, KERNEL_SIZE,
                          DROPOUT_RATE, LEARNING_RATE)

        X_tr_r = X_tr.reshape(-1, X_tr.shape[1], 1)
        X_vl_r = X_vl.reshape(-1, X_vl.shape[1], 1)

        history = model.fit(
            X_tr_r, y_tr,
            validation_data=(X_vl_r, y_vl),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[
                EarlyStopping(monitor='val_loss', patience=PATIENCE,
                              restore_best_weights=True),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                  patience=max(PATIENCE // 2, 2), min_lr=1e-6),
            ],
            verbose=0
        )

        preds = model.predict(
            X_te.reshape(-1, X_te.shape[1], 1),
            batch_size=2048, verbose=0
        )
        elapsed = time.time() - t0

        m = compute_metrics(y_te, preds)
        m['Time_s'] = round(elapsed, 2)
        results.append({'Patient': rid, 'Horizon': horizon, **m})

        if horizon > 1:
            per_step_results[(rid, horizon)] = compute_per_step_metrics(y_te, preds)

        saved_preds[(rid, horizon)] = (y_te.copy(), preds.copy())
        training_histories[(rid, horizon)] = {
            'loss': history.history['loss'],
            'val_loss': history.history['val_loss'],
        }

        del model
        gc.collect()

        tqdm.write(
            f'  Patient {rid} H={horizon:2d} | '
            f'R²={m["R2"]:.4f}  RMSE={m["RMSE"]:.4f}  MAE={m["MAE"]:.4f}  '
            f'({elapsed:.1f}s)'
        )

    tf.keras.backend.clear_session()
    gc.collect()

print(f'\nAll {len(results)} experiments complete.')

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────────
df_results = pd.DataFrame(results)
df_results.to_csv(CSV_NAME, index=False)
print(f'Results saved to {CSV_NAME}')
display(df_results)

## 9 — Compare: Baseline vs Optimized

In [ ]:
# ── Load baseline results for comparison ─────────────────────────────────────
df_baseline = pd.read_csv('Figures/results/tcn_mo_ours_config_results.csv')
df_optimized = df_results.copy()

print('Mean Metrics Comparison: Baseline (Ours) vs Optimized')
print('=' * 70)
print(f'{"H":>3s}  {"RMSE(base)":>11s} {"RMSE(opt)":>11s} {"Δ%":>7s}  '
      f'{"R²(base)":>9s} {"R²(opt)":>9s} {"Δ%":>7s}')
print('-' * 70)

for h in HORIZONS:
    b = df_baseline[df_baseline.Horizon == h]
    o = df_optimized[df_optimized.Horizon == h]
    rmse_b, rmse_o = b.RMSE.mean(), o.RMSE.mean()
    r2_b, r2_o = b.R2.mean(), o.R2.mean()
    rmse_pct = (rmse_o - rmse_b) / rmse_b * 100
    r2_pct = (r2_o - r2_b) / r2_b * 100
    print(f'{h:3d}  {rmse_b:11.6f} {rmse_o:11.6f} {rmse_pct:+6.1f}%  '
          f'{r2_b:9.4f} {r2_o:9.4f} {r2_pct:+6.2f}%')

print('-' * 70)
print('Negative RMSE Δ% = improvement (lower is better)')
print('Positive R² Δ%   = improvement (higher is better)')

In [ ]:
# ── Win rate analysis ────────────────────────────────────────────────────────
print('\nPer-Patient Win Rates (Optimized vs Baseline)')
print('=' * 50)
for h in HORIZONS:
    b = df_baseline[df_baseline.Horizon == h].sort_values('Patient').RMSE.values
    o = df_optimized[df_optimized.Horizon == h].sort_values('Patient').RMSE.values
    wins = (o < b).sum()
    total = len(b)
    print(f'  H={h:2d}: Optimized wins {wins}/{total} patients ({wins/total*100:.0f}%)')

In [ ]:
# ── Statistical significance (Wilcoxon signed-rank test) ─────────────────────
from scipy.stats import wilcoxon

print('\nWilcoxon Signed-Rank Test (paired, two-sided)')
print('H₀: no difference between baseline and optimized RMSE')
print('=' * 50)
for h in HORIZONS:
    b = df_baseline[df_baseline.Horizon == h].sort_values('Patient').RMSE.values
    o = df_optimized[df_optimized.Horizon == h].sort_values('Patient').RMSE.values
    stat, p = wilcoxon(b, o)
    sig = '✓ significant' if p < 0.05 else '✗ not significant'
    print(f'  H={h:2d}: p={p:.4f} → {sig}')

In [ ]:
# ── Comparison bar charts ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, metric, direction in zip(axes, ['RMSE', 'MAE', 'R2'], ['↓', '↓', '↑']):
    base_means = [df_baseline[df_baseline.Horizon == h][metric].mean() for h in HORIZONS]
    opt_means  = [df_optimized[df_optimized.Horizon == h][metric].mean() for h in HORIZONS]

    x = np.arange(len(HORIZONS))
    w = 0.35
    ax.bar(x - w/2, base_means, w, label='Baseline (Ours)', color='#90CAF9', edgecolor='white')
    ax.bar(x + w/2, opt_means, w, label='Optimized', color='#1565C0', edgecolor='white')
    ax.set_xlabel('Forecast Horizon')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} {direction}')
    ax.set_xticks(x)
    ax.set_xticklabels(HORIZONS)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Baseline vs Optimized Configuration', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/comparison_baseline_vs_optimized.png', dpi=200, bbox_inches='tight')
plt.show()

## 10 — Summary & Next Steps

In [ ]:
print('\n' + '=' * 80)
print(f'HYPERPARAMETER OPTIMIZATION COMPLETE')
print('=' * 80)
print(f'\nSearch:')
print(f'  Trials         : {len(study.trials)}')
print(f'  Best trial     : #{study.best_trial.number}')
print(f'  Best pilot RMSE: {study.best_value:.6f}')
print(f'\nBest Configuration:')
for k, v in study.best_params.items():
    print(f'  {k:>12s} = {v}')
print(f'\nFull Evaluation ({CONFIG_NAME}):')
print(f'  Patients       : {len(PATIENTS)}')
print(f'  Horizons       : {HORIZONS}')
print(f'  Results CSV    : {CSV_NAME}')
print(f'  Plots          : {PLOT_DIR}/')
print(f'\nComparison vs Baseline:')
for h in HORIZONS:
    b_rmse = df_baseline[df_baseline.Horizon == h].RMSE.mean()
    o_rmse = df_optimized[df_optimized.Horizon == h].RMSE.mean()
    pct = (o_rmse - b_rmse) / b_rmse * 100
    print(f'  H={h:2d}: RMSE {b_rmse:.4f} → {o_rmse:.4f} ({pct:+.1f}%)')
print('\n' + '=' * 80)

---

### Next Steps

1. **Phase 2 search:** Add activation function, weight decay, lookback window to search space
2. **Per-horizon optimization:** Run separate studies for each horizon if H=1 and H=20 need different configs
3. **Cross-validation:** K-fold patient-wise CV on best config for robustness estimate
4. **Ensemble:** Combine top-3 configs for potential further gains
5. **Report:** Update LaTeX comparison report with optimized results